# Example: Breadth-First Search (BFS) and Depth First Search (DFS) on some Simple Trees and Graphs
This example will familiarize students with [Breadth-First Search](https://en.wikipedia.org/wiki/Breadth-first_search) and [Depth-First Search](https://en.wikipedia.org/wiki/Depth-first_search) graph traversal on a simple graph (and tree) example. 

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [1]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="480"/>
    </center>
</div>

## Task 1: Build a graph model instance for an example graph using an edge list
In this task, we'll build [a `MySimpleDirectedGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel) from an edge list file stored in the `data` folder.

### Adjacency Matrix versus Edge List
A graph $\mathcal{G}=\left(\mathcal{V},\mathcal{E}\right)$ can be constructed from an [Adjacency Matrix](https://en.wikipedia.org/wiki/Adjacency_matrix) $\mathbf{A}$, which is a $\dim\mathcal{V}\times\dim\mathcal{V}$ square matrix. However, this is only suitable for small graphs because $\mathbf{A}$ has a high memory overhead (if stored as `64-bit` values). 
* For example, consider a graph with $\dim\mathcal{V}$ = 100000 would require `80 GB` of memory to store in the worst case, which is more than most common machines

In [2]:
𝒱 = 100000;
memory_reqd = (𝒱^2)*8*(1/(1e9)) # 8 x bytes (64-bits) for each entry = units GB

80.0

Instead, a lower memory representation is an [Edge list representation](https://en.wikipedia.org/wiki/Edge_list). In the [Edge list representation](https://en.wikipedia.org/wiki/Edge_list), only the edge information is stored (typically) in a comma-separated value (CSV) file in which each record holds an edge in the graph, and the fields contain `source, target, weight` data for the edge.

We've built [the `MyGraphEdgeModel(...))` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels) which takes a file path as an argument, and returns a dictionary of [`MyGraphEdgeModel` instances](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyGraphEdgeModel) to hold this information. Let's load up an edge list. 

First, set the path to the edge list file:

In [3]:
path_to_edge_file = joinpath(_PATH_TO_DATA, "SimpleGraph.txt"); # this will load the graph shown above

Next, let's construct our dictionary of `edgemodels,` where the data for the edges (source id, target id, and weight) is stored in [a `MyGraphEdgeModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyGraphEdgeModel).

Because each edge file could have a different record structure, let's define a __callback function__ that is called for each line of the edge file. This function will parse the record in the edge file, and return the source id, target id, and weight as a tuple. Note that the internal data structure uses the `Any` data type for the weight, so this can be a string, float, integer or an object, etc.

In this case, we'll keep it simple: we'll just return the weight as a float.

In [ ]:
function edgerecordparser(record::String, delim::Char=',')
    
    fields = split(record, delim) # this assumes a record of the form "source,target,weight"
    if length(fields) < 3
        return nothing
    end

    source = parse(Int, fields[1]) # source id
    target = parse(Int, fields[2]) # target id
    weight = parse(Float64, fields[3]) # edge weight
    
    return (source, target, weight)
end

edgerecordparser (generic function with 2 methods)

We have our path to the edge list file, and the callback function `edgerecordparser` defined. Now we can create our edgemodel dictionary by calling [the  `MyGraphEdgeModels(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels).

Let's save our edge models in the `myedgemodels::Dict{Int64, MyGraphEdgeModel}` dictionary. The keys will be the edge ids (which we can assume are unique), and the values will be the corresponding `MyGraphEdgeModel` instances. Here, we'll use the line index as the edge id.

In [5]:
myedgemodels = MyGraphEdgeModels(path_to_edge_file, edgerecordparser, delim=',', comment='#')

Dict{Int64, MyGraphEdgeModel} with 7 entries:
  0 => MyGraphEdgeModel(0, 1, 2, 10.0)
  4 => MyGraphEdgeModel(4, 3, 5, 6.0)
  5 => MyGraphEdgeModel(5, 4, 6, 1.0)
  6 => MyGraphEdgeModel(6, 5, 4, 1.0)
  2 => MyGraphEdgeModel(2, 2, 3, 2.0)
  3 => MyGraphEdgeModel(3, 2, 4, 100.0)
  1 => MyGraphEdgeModel(1, 1, 3, 100.0)

What are the fields in [the `MyGraphEdgeModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyGraphEdgeModel)?

In [7]:
typeof(myedgemodels[1]) |> m-> fieldnames(m)

(:id, :source, :target, :weight)

Finally, now that we have the `myedgemodels` dictionary, we can build a graph instance. Since this is a directed graph, we'll construct [a `MySimpleDirectedGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel) using [a `build(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build).

Let's save our graph model in the `directedgraphmodel::MySimpleDirectedGraphModel` variable.

In [9]:
directedgraphmodel = build(MySimpleDirectedGraphModel, myedgemodels)

MySimpleDirectedGraphModel(Dict{Int64, MyGraphNodeModel}(5 => MyGraphNodeModel(5, nothing), 4 => MyGraphNodeModel(4, nothing), 6 => MyGraphNodeModel(6, nothing), 2 => MyGraphNodeModel(2, nothing), 3 => MyGraphNodeModel(3, nothing), 1 => MyGraphNodeModel(1, nothing)), Dict((2, 4) => 100, (1, 2) => 10, (1, 3) => 100, (4, 6) => 1, (3, 5) => 6, (5, 4) => 1, (2, 3) => 2), Dict{Int64, Set{Int64}}(5 => Set([4]), 4 => Set([6]), 6 => Set(), 2 => Set([4, 3]), 3 => Set([5]), 1 => Set([2, 3])))

What fields are in the `MySimpleDirectedGraphModel` type?

In [10]:
typeof(directedgraphmodel) |> m-> fieldnames(m)

(:nodes, :edges, :children)